In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip -q "/content/drive/MyDrive/Original Dataset.zip" -d /content/pumpkin_data

In [ ]:
import os
for root, dirs, files_ in os.walk('/content/pumpkin_data'):
    level = root.replace('/content/pumpkin_data', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        for f in files_[:3]:
            print(f'{indent}  {f}')

pumpkin_data/
  Original Dataset/
    Healthy Leaf/
    Mosaic Disease/
    Bacterial Leaf Spot/
    Downy Mildew/
    Powdery_Mildew/


In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, datasets, models
from torch.utils.data import DataLoader, Subset, random_split
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return running_loss / total, correct / total

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, 5)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

In [ ]:
NUM_EPOCHS = 50
best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    start = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    elapsed = time.time() - start

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_resnet50_v2.pth')
        marker = "  <- saved (best so far)"
    else:
        marker = ""

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Train loss: {train_loss:.4f}, acc: {train_acc:.4f} | "
          f"Val loss: {val_loss:.4f}, acc: {val_acc:.4f} | "
          f"({elapsed:.1f}s){marker}")

print(f"\nBest val accuracy achieved: {best_val_acc:.4f}")

Epoch 1/50 | Train loss: 1.5250, acc: 0.4581 | Val loss: 1.3921, acc: 0.6250 | (41.2s)  <- saved (best so far)
Epoch 2/50 | Train loss: 1.2171, acc: 0.6981 | Val loss: 0.8800, acc: 0.7450 | (39.0s)  <- saved (best so far)
Epoch 3/50 | Train loss: 0.8549, acc: 0.7575 | Val loss: 0.6469, acc: 0.8000 | (41.8s)  <- saved (best so far)
Epoch 4/50 | Train loss: 0.6633, acc: 0.7863 | Val loss: 0.5246, acc: 0.8350 | (40.3s)  <- saved (best so far)
Epoch 5/50 | Train loss: 0.5935, acc: 0.8019 | Val loss: 0.4702, acc: 0.8550 | (41.1s)  <- saved (best so far)
Epoch 6/50 | Train loss: 0.5388, acc: 0.8213 | Val loss: 0.4529, acc: 0.8400 | (41.3s)
Epoch 7/50 | Train loss: 0.5180, acc: 0.8187 | Val loss: 0.4417, acc: 0.8550 | (42.4s)
Epoch 8/50 | Train loss: 0.4708, acc: 0.8475 | Val loss: 0.4286, acc: 0.8550 | (41.0s)
Epoch 9/50 | Train loss: 0.4576, acc: 0.8506 | Val loss: 0.4239, acc: 0.8450 | (41.1s)
Epoch 10/50 | Train loss: 0.4287, acc: 0.8506 | Val loss: 0.4133, acc: 0.8550 | (41.0s)
Epoch 11/

In [ ]:
model.load_state_dict(torch.load('best_resnet50_v2.pth'))
test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f"Test accuracy (best checkpoint): {test_acc:.4f}")

Test accuracy (best checkpoint): 0.8800
